# Heat Pump Stock Rally Prediction — V10
Heterogeneous Stacking: XGB + LGB + RF + TabPFN → XGB Meta-Learner

In [ ]:
# Cell 1 — Install packages & imports
import subprocess, sys

def _pip(*pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *pkgs])

_pip("yfinance", "ta", "optuna", "xgboost", "lightgbm", "scikit-learn",
     "shap", "matplotlib", "pandas", "numpy", "requests", "vaderSentiment",
     "tabpfn")

import os, warnings, logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from concurrent.futures import ThreadPoolExecutor
from sklearn.ensemble import RandomForestClassifier
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import precision_score, recall_score, f1_score, average_precision_score, precision_recall_curve
import xgboost as xgb
import lightgbm as lgb
import optuna
import shap
import ta
import yfinance as yf

try:
    from tabpfn import TabPFNClassifier
    TABPFN_AVAILABLE = True
except ImportError:
    TABPFN_AVAILABLE = False

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
logging.getLogger('lightgbm').setLevel(logging.ERROR)
logging.getLogger('xgboost').setLevel(logging.ERROR)

try:
    import mlflow
    mlflow.autolog(disable=True)
except ImportError:
    pass

print('All imports OK')

In [ ]:
# Cell 2 — Configuration

# ── Dates ──────────────────────────────────────────────────────────────────
START_DATE = '2020-01-01'
END_DATE   = '2025-06-01'

# ── Ticker universe + sector mapping ───────────────────────────────────────
TICKERS_BY_SECTOR = {
    'heat_pump':  ['NIBE-B.ST', '6367.T', 'CARR', 'TT', 'JCI', 'LII', 'AOS', 'WSO', '6503.T', 'AALB.AS'],
    'tech':       ['AAPL', 'MSFT', 'NVDA', 'CRM', 'CSCO', 'INTC', 'IBM'],
    'finance':    ['JPM', 'GS', 'AXP', 'V'],
    'healthcare': ['JNJ', 'UNH', 'MRK', 'AMGN'],
    'consumer':   ['KO', 'MCD', 'NKE', 'PG', 'WMT', 'HD', 'DIS'],
    'industrial': ['CAT', 'BA', 'HON', 'MMM', 'SHW', 'TRV', 'DOW'],
    'energy':     ['CVX'],
    'crypto':     ['BTC-USD', 'ETH-USD', 'BNB-USD', 'SOL-USD', 'XRP-USD', 'ADA-USD'],
}

# Alphabetically sorted sector labels (0-based)
SECTOR_LABELS = {s: i for i, s in enumerate(sorted(TICKERS_BY_SECTOR.keys()))}
# consumer=0, crypto=1, energy=2, finance=3, healthcare=4, heat_pump=5, industrial=6, tech=7

TICKER_TO_SECTOR = {t: s for s, tl in TICKERS_BY_SECTOR.items() for t in tl}
ALL_TICKERS = [t for tl in TICKERS_BY_SECTOR.values() for t in tl]

# Company display names
COMPANY_NAMES = {
    'NIBE-B.ST': 'NIBE', '6367.T': 'Daikin', 'CARR': 'Carrier', 'TT': 'Trane Tech',
    'JCI': 'Johnson Controls', 'LII': 'Lennox', 'AOS': 'A.O. Smith', 'WSO': 'Watsco',
    '6503.T': 'Mitsubishi', 'AALB.AS': 'Aalberts',
    'AAPL': 'Apple', 'MSFT': 'Microsoft', 'NVDA': 'NVIDIA', 'CRM': 'Salesforce',
    'CSCO': 'Cisco', 'INTC': 'Intel', 'IBM': 'IBM',
    'JPM': 'JPMorgan', 'GS': 'Goldman Sachs', 'AXP': 'AmEx', 'V': 'Visa',
    'JNJ': 'J&J', 'UNH': 'UnitedHealth', 'MRK': 'Merck', 'AMGN': 'Amgen',
    'KO': 'Coca-Cola', 'MCD': "McDonald's", 'NKE': 'Nike', 'PG': 'P&G',
    'WMT': 'Walmart', 'HD': 'Home Depot', 'DIS': 'Disney',
    'CAT': 'Caterpillar', 'BA': 'Boeing', 'HON': 'Honeywell', 'MMM': '3M',
    'SHW': 'Sherwin-Williams', 'TRV': 'Travelers', 'DOW': 'Dow Inc.',
    'CVX': 'Chevron',
    'BTC-USD': 'Bitcoin', 'ETH-USD': 'Ethereum', 'BNB-USD': 'BNB',
    'SOL-USD': 'Solana', 'XRP-USD': 'XRP', 'ADA-USD': 'Cardano',
}

# ── Pipeline constants ──────────────────────────────────────────────────────
RETURN_WINDOW       = 10    # Rolling window for rally detection
RALLY_THRESHOLD     = 0.07  # 7% cumulative return
LEAD_DAYS           = 3     # Pre-rally signal window
ENTRY_DAYS          = 3     # Entry signal window into rally
N_WF_SPLITS         = 5     # Walk-forward CV folds
EARLY_STOPPING_ROUNDS = 30
N_OPTUNA_TRIALS     = 50
N_META_TRIALS       = 30
RANDOM_STATE        = 42
N_WORKERS           = os.cpu_count()
CONSECUTIVE_DAYS    = 1
SIGNAL_COOLDOWN_DAYS = 3

# ── Indicator parameter grids ───────────────────────────────────────────────
RSI_WINDOWS  = [7, 10, 14, 21]
BB_WINDOWS   = [15, 20, 25]
SMA_WINDOWS  = [30, 50, 70]

# ── Seed params from V8 (first Optuna trial) ───────────────────────────────
SEED_PARAMS = dict(
    rsi_window=10, bb_window=15, sma_window=70,
    grow_policy='lossguide', max_leaves=811, max_bin=128,
    max_depth=8, min_child_weight=5, gamma=5.423,
    reg_alpha=1.3e-6, reg_lambda=0.014, learning_rate=0.064,
    n_estimators=182, subsample=0.651, colsample_bytree=0.659,
    focal_gamma=0.748, focal_alpha=0.300,
)

# ── News sentiment (disabled) ───────────────────────────────────────────────
USE_NEWS_SENTIMENT = False
NEWS_FEATURE_COLS  = ['sent_compound', 'sent_pos', 'sent_neg', 'news_count']
SECTOR_KEYWORDS = {
    'heat_pump':  ['heat pump', 'HVAC', 'refrigerant', 'air conditioning', 'Daikin', 'Carrier'],
    'tech':       ['semiconductor', 'AI chip', 'cloud computing', 'software'],
    'finance':    ['interest rate', 'bank earnings', 'Fed', 'credit'],
    'healthcare': ['FDA approval', 'clinical trial', 'pharma', 'biotech'],
    'consumer':   ['consumer spending', 'retail sales', 'inflation'],
    'industrial': ['manufacturing PMI', 'industrial output', 'capex'],
    'energy':     ['oil price', 'crude', 'OPEC', 'natural gas'],
    'crypto':     ['Bitcoin', 'Ethereum', 'crypto regulation', 'DeFi'],
}

# ── Feature column builder ──────────────────────────────────────────────────
def build_feature_cols(rsi_w, bb_w, sma_w):
    """Returns exactly 24 feature column names for given window params."""
    return [
        # A. Base indicators (7)
        f'rsi_{rsi_w}',
        f'bb_pband_{bb_w}',
        'macd_diff',
        'vol_stress',
        'drawdown',
        'adx',
        'vol_ratio',
        # B. Interaction (1)
        f'bb_x_rsi_{bb_w}_{rsi_w}',
        # C. Trajectories / deltas (5)
        f'rsi_delta_3d_{rsi_w}',
        f'bb_slope_5d_{bb_w}',
        f'bb_delta_3d_{bb_w}',
        'adx_delta_3d',
        'momentum_accel',
        # D. Multi-timeframe (2)
        f'rsi_weekly_{rsi_w}',
        f'sma_cross_20_{sma_w}',
        # E. Regime (3)
        'close_vs_sma200',
        'sma200_delta_5d',
        'drawdown_252d',
        # F. Market breadth (1)
        f'market_breadth_{sma_w}',
        # G. Volume breakout (1)
        'volume_zscore',
        # H. Macro / cross-asset (3)
        f'sector_avg_rsi_{rsi_w}',
        'btc_momentum',
        'month',
        # I. Sector encoding (1)
        'sector_id',
    ]

def build_rename_map(rsi_w, bb_w, sma_w):
    """Human-readable display names for SHAP plots."""
    return {
        f'rsi_{rsi_w}':              f'RSI ({rsi_w}d)',
        f'bb_pband_{bb_w}':          f'Bollinger %B ({bb_w}d)',
        f'bb_x_rsi_{bb_w}_{rsi_w}': f'BB({bb_w}) × RSI({rsi_w})',
        f'sma_cross_20_{sma_w}':     f'SMA20 / SMA{sma_w}',
        f'market_breadth_{sma_w}':   f'Breadth (SMA{sma_w})',
        f'sector_avg_rsi_{rsi_w}':   f'Sector Avg RSI ({rsi_w}d)',
        f'rsi_delta_3d_{rsi_w}':     f'RSI Δ3d ({rsi_w}d)',
        f'bb_slope_5d_{bb_w}':       f'BB Slope 5d ({bb_w}d)',
        f'bb_delta_3d_{bb_w}':       f'BB Δ3d ({bb_w}d)',
        f'rsi_weekly_{rsi_w}':       f'RSI Weekly ({rsi_w}d)',
        'macd_diff':       'MACD Histogram',
        'vol_stress':      'Vol Stress',
        'drawdown':        'Drawdown 60d',
        'adx':             'ADX',
        'vol_ratio':       'Vol Ratio 5/20d',
        'adx_delta_3d':    'ADX Δ3d',
        'momentum_accel':  'Momentum Accel',
        'close_vs_sma200': 'Close / SMA200',
        'sma200_delta_5d': 'SMA200 Ratio Δ 5d',
        'drawdown_252d':   'Drawdown 252d',
        'volume_zscore':   'Volume Z-Score',
        'btc_momentum':    'BTC Momentum',
        'month':           'Month',
        'sector_id':       'Sector ID',
    }

print('Configuration loaded.')
print(f'Total tickers: {len(ALL_TICKERS)}')
print(f'Sector labels: {SECTOR_LABELS}')

In [ ]:
# Cell 3 — Helpers

def _strip_tz(series):
    """Normalize DatetimeIndex or datetime Series to tz-naive midnight."""
    if hasattr(series, 'dt'):
        if series.dt.tz is not None:
            series = series.dt.tz_convert(None)
        return series.dt.normalize()
    if isinstance(series, pd.DatetimeIndex):
        if series.tz is not None:
            series = series.tz_convert(None)
        return series.normalize()
    return series


def make_focal_objective(gamma, alpha):
    """
    Returns a custom objective function for XGBoost/LightGBM implementing
    Focal Loss with given focusing parameter gamma and class weight alpha.

    L = -alpha_t * (1 - p_t)^gamma * log(p_t)
    where p_t = p if y=1 else (1-p), alpha_t = alpha if y=1 else (1-alpha)
    """
    def focal_obj(y_pred, dtrain):
        y_true = dtrain.get_label()
        p = 1.0 / (1.0 + np.exp(-y_pred))          # sigmoid
        p = np.clip(p, 1e-7, 1 - 1e-7)

        # Gradient
        grad = (
            y_true * alpha * (1 - p) ** gamma
            * (gamma * p * np.log(p) + p - 1)
            + (1 - y_true) * (1 - alpha) * p ** gamma
            * (p - gamma * (1 - p) * np.log(1 - p))
        )

        # Hessian (weighted logistic approximation)
        p_t   = np.where(y_true == 1, p, 1 - p)
        a_t   = np.where(y_true == 1, alpha, 1 - alpha)
        hess  = np.maximum(a_t * (1 - p_t) ** gamma * p * (1 - p), 1e-7)
        return grad, hess

    return focal_obj


def make_focal_objective_lgb(gamma, alpha):
    """LightGBM version of focal loss objective (raw logit input)."""
    def focal_obj_lgb(y_pred, dataset):
        y_true = dataset.get_label()
        p = 1.0 / (1.0 + np.exp(-y_pred))
        p = np.clip(p, 1e-7, 1 - 1e-7)

        grad = (
            y_true * alpha * (1 - p) ** gamma
            * (gamma * p * np.log(p) + p - 1)
            + (1 - y_true) * (1 - alpha) * p ** gamma
            * (p - gamma * (1 - p) * np.log(1 - p))
        )

        p_t  = np.where(y_true == 1, p, 1 - p)
        a_t  = np.where(y_true == 1, alpha, 1 - alpha)
        hess = np.maximum(a_t * (1 - p_t) ** gamma * p * (1 - p), 1e-7)
        return grad, hess

    return focal_obj_lgb


print('Helpers defined.')

In [ ]:
# Cell 4 — 1: Load stock data

def load_stock_data(tickers=ALL_TICKERS, start=START_DATE, end=END_DATE):
    """
    Single bulk yfinance download with threads=False to avoid data corruption bug.
    Returns a DataFrame with columns [Date, close, volume, ticker, company].
    """
    print(f'Downloading {len(tickers)} tickers from {start} to {end}...')
    raw = yf.download(
        tickers,
        start=start,
        end=end,
        auto_adjust=True,
        threads=False,   # CRITICAL: threads=True corrupts multi-ticker data
        progress=False,
    )

    frames = []
    for ticker in tickers:
        try:
            if isinstance(raw.columns, pd.MultiIndex):
                close  = raw['Close'][ticker].dropna()
                volume = raw['Volume'][ticker].reindex(close.index).fillna(0)
            else:
                close  = raw['Close'].dropna()
                volume = raw['Volume'].reindex(close.index).fillna(0)

            if len(close) < 100:
                print(f'  Skipping {ticker}: only {len(close)} rows')
                continue

            df = pd.DataFrame({'close': close, 'volume': volume})
            df.index = _strip_tz(df.index)
            df = df.reset_index().rename(columns={'index': 'Date', 'Price': 'Date'})
            if 'Date' not in df.columns:
                df = df.rename(columns={df.columns[0]: 'Date'})
            df['Date']    = _strip_tz(df['Date'])
            df['ticker']  = ticker
            df['company'] = COMPANY_NAMES.get(ticker, ticker)
            frames.append(df)
        except Exception as e:
            print(f'  Error {ticker}: {e}')

    result = pd.concat(frames, ignore_index=True)
    result['Date'] = pd.to_datetime(result['Date'])
    print(f'Loaded {result["ticker"].nunique()} tickers, {len(result):,} rows.')
    return result

In [ ]:
# Cell 5 — 2: Target vector

def _create_target_one_ticker(df_ticker):
    """
    For a single ticker DataFrame (sorted by Date), compute:
      - rally[]: 1 where the 10-day cumulative return >= 7%
      - target[]: 1 for LEAD_DAYS before rally start + ENTRY_DAYS into rally
    """
    close = df_ticker['close'].values.astype(np.float64)
    n = len(close)
    window = RETURN_WINDOW

    # Rolling cumulative return (product of (1+daily_ret) over window)
    daily_ret = np.full(n, np.nan)
    daily_ret[1:] = close[1:] / close[:-1] - 1.0

    cum_ret = np.full(n, np.nan)
    for i in range(window - 1, n):
        product = 1.0
        for j in range(i - window + 1, i + 1):
            if not np.isnan(daily_ret[j]):
                product *= (1.0 + daily_ret[j])
        cum_ret[i] = product - 1.0

    # Mark rally periods (backwards from qualifying end-day)
    rally = np.zeros(n, dtype=np.int8)
    for end_idx in range(n):
        if not np.isnan(cum_ret[end_idx]) and cum_ret[end_idx] >= RALLY_THRESHOLD:
            start_idx = max(0, end_idx - window + 1)
            rally[start_idx:end_idx + 1] = 1

    # Signal window: pre-rally + entry
    target = np.zeros(n, dtype=np.int8)
    i = 0
    while i < n:
        if rally[i] == 1 and (i == 0 or rally[i - 1] == 0):
            # Start of a new rally segment
            pre_start = max(0, i - LEAD_DAYS)
            target[pre_start:i]                   = 1  # pre-rally
            target[i:min(n, i + ENTRY_DAYS)]      = 1  # entry
        i += 1

    return rally, target


def create_target(df):
    """Add 'rally' and 'target' columns to full DataFrame, parallelised per ticker."""
    df = df.sort_values(['ticker', 'Date']).copy()
    rally_col  = np.zeros(len(df), dtype=np.int8)
    target_col = np.zeros(len(df), dtype=np.int8)

    def process(args):
        idx, sub = args
        r, t = _create_target_one_ticker(sub.reset_index(drop=True))
        return idx, r, t

    groups = list(df.groupby('ticker'))
    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        results = list(ex.map(process, groups))

    for ticker, sub in groups:
        pass  # just to reuse the loop below

    # Re-assign in correct row order
    for ticker_name, sub in df.groupby('ticker'):
        r, t = _create_target_one_ticker(sub.reset_index(drop=True))
        df.loc[sub.index, 'rally']  = r
        df.loc[sub.index, 'target'] = t

    pos_rate = df['target'].mean()
    print(f'Target created. Positive rate: {pos_rate:.1%}')
    return df

In [ ]:
# Cell 6 — 3: Technical indicators

def _linreg_slope(values):
    """Slope of OLS line through 'values' (length 5 expected)."""
    n = len(values)
    x = np.arange(n, dtype=float) - np.mean(np.arange(n, dtype=float))
    return np.dot(x, values) / np.dot(x, x)


def _compute_indicators_one_ticker(df_t):
    """
    Compute all indicator variants for one ticker.
    Returns a DataFrame with all precomputed columns.
    """
    df = df_t.sort_values('Date').copy().reset_index(drop=True)
    close  = df['close']
    volume = df['volume']

    # ── Log return for vol_stress ───────────────────────────────────────────
    log_ret = np.log(close / close.shift(1))

    # ── Vol stress ─────────────────────────────────────────────────────────
    df['vol_stress'] = (
        log_ret.rolling(10, min_periods=5).std() /
        log_ret.rolling(60, min_periods=20).std()
    )

    # ── Drawdown 60d ───────────────────────────────────────────────────────
    rolling_max_60 = close.rolling(60, min_periods=1).max()
    df['drawdown'] = (close - rolling_max_60) / rolling_max_60

    # ── MACD ───────────────────────────────────────────────────────────────
    df['macd_diff'] = ta.trend.MACD(
        close, window_slow=26, window_fast=12, window_sign=9
    ).macd_diff()

    # ── ADX (synthetic H/L from close) ────────────────────────────────────
    df['adx'] = ta.trend.ADXIndicator(
        high=close * 1.01, low=close * 0.99, close=close, window=14
    ).adx()
    df['adx_delta_3d'] = df['adx'].diff(3)

    # ── Volume features ────────────────────────────────────────────────────
    vol_5d_mean  = volume.rolling(5,  min_periods=1).mean()
    vol_20d_mean = volume.rolling(20, min_periods=5).mean()
    vol_60d_mean = volume.rolling(60, min_periods=20).mean()
    vol_60d_std  = volume.rolling(60, min_periods=20).std()

    df['vol_ratio']     = vol_5d_mean / (vol_20d_mean + 1e-10)
    df['volume_zscore'] = (vol_5d_mean - vol_60d_mean) / (vol_60d_std + 1e-10)

    # ── Momentum 20d (needed for momentum_accel + cross-asset) ─────────────
    df['momentum_20d'] = close.pct_change(20)
    df['momentum_accel'] = df['momentum_20d'].diff(5)

    # ── Regime features ────────────────────────────────────────────────────
    sma200 = close.rolling(200, min_periods=150).mean()
    df['close_vs_sma200'] = close / sma200
    df['sma200_delta_5d'] = df['close_vs_sma200'].diff(5)

    rolling_max_252 = close.rolling(252, min_periods=60).max()
    df['drawdown_252d'] = (close - rolling_max_252) / (rolling_max_252 + 1e-10)

    # ── RSI — all windows ──────────────────────────────────────────────────
    for w in RSI_WINDOWS:
        df[f'rsi_{w}'] = ta.momentum.rsi(close, window=w)
        df[f'rsi_delta_3d_{w}'] = df[f'rsi_{w}'].diff(3)

        # Weekly RSI (resample → ffill)
        try:
            weekly = close.copy()
            weekly.index = df['Date']
            weekly_c = weekly.resample('W-FRI').last().dropna()
            if len(weekly_c) >= w + 5:
                rsi_wk = ta.momentum.rsi(weekly_c, window=w)
                rsi_wk = rsi_wk.reindex(df['Date'], method='ffill')
                df[f'rsi_weekly_{w}'] = rsi_wk.values
            else:
                df[f'rsi_weekly_{w}'] = np.nan
        except Exception:
            df[f'rsi_weekly_{w}'] = np.nan

    # ── Bollinger Bands — all windows ──────────────────────────────────────
    for w in BB_WINDOWS:
        bb = ta.volatility.BollingerBands(close, window=w, window_dev=2)
        df[f'bb_pband_{w}'] = bb.bollinger_pband()
        df[f'bb_delta_3d_{w}'] = df[f'bb_pband_{w}'].diff(3)

        # Bollinger slope (rolling OLS over 5 points)
        df[f'bb_slope_5d_{w}'] = (
            df[f'bb_pband_{w}']
            .rolling(5)
            .apply(_linreg_slope, raw=True)
        )

    # ── Interaction: BB × RSI (all combinations) ──────────────────────────
    for bw in BB_WINDOWS:
        for rw in RSI_WINDOWS:
            df[f'bb_x_rsi_{bw}_{rw}'] = (
                df[f'bb_pband_{bw}'] * (df[f'rsi_{rw}'] / 100.0)
            )

    # ── SMA cross + sma_ratio (for market breadth) ─────────────────────────
    sma20 = close.rolling(20, min_periods=15).mean()
    for sw in SMA_WINDOWS:
        sma_sw = close.rolling(sw, min_periods=int(0.6 * sw)).mean()
        df[f'sma_cross_20_{sw}'] = sma20 / (sma_sw + 1e-10)
        df[f'sma_ratio_{sw}']    = close / (sma_sw + 1e-10)

    return df


def add_technical_indicators(df):
    """Compute all indicators for all tickers in parallel."""
    groups = list(df.groupby('ticker'))
    with ThreadPoolExecutor(max_workers=N_WORKERS) as ex:
        results = list(ex.map(lambda g: _compute_indicators_one_ticker(g[1]), groups))
    out = pd.concat(results, ignore_index=True)
    print(f'Indicators computed. Shape: {out.shape}')
    return out

In [ ]:
# Cell 7 — 4: News sentiment (GDELT + VADER) — DISABLED

def _gdelt_chunk(keyword, date_from, date_to, max_articles=200):
    """Fetch one 90-day GDELT chunk for a keyword. Returns list of (date, headline)."""
    import requests, time
    url = (
        f'https://api.gdeltproject.org/api/v2/doc/doc?query={requests.utils.quote(keyword)}'
        f'&mode=artlist&maxrecords={max_articles}'
        f'&startdatetime={date_from.strftime("%Y%m%d%H%M%S")}'
        f'&enddatetime={date_to.strftime("%Y%m%d%H%M%S")}'
        f'&format=json'
    )
    for attempt in range(3):
        try:
            r = requests.get(url, timeout=30)
            if r.status_code == 429:
                time.sleep(60 * (attempt + 1))
                continue
            data = r.json()
            articles = data.get('articles', [])
            return [(a.get('seendate', ''), a.get('title', '')) for a in articles]
        except Exception:
            time.sleep(10)
    return []


def _score_and_aggregate(records, sector, business_days):
    """Score headlines with VADER, aggregate per business day."""
    from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
    sia = SentimentIntensityAnalyzer()
    rows = []
    for datestr, headline in records:
        try:
            date = pd.to_datetime(str(datestr)[:8], format='%Y%m%d').normalize()
        except Exception:
            continue
        sc = sia.polarity_scores(headline)
        rows.append({'Date': date, 'compound': sc['compound'],
                     'pos': sc['pos'], 'neg': sc['neg']})
    if not rows:
        return pd.DataFrame()
    sdf = pd.DataFrame(rows)
    agg = sdf.groupby('Date').agg(
        sent_compound=('compound', 'mean'),
        sent_pos=('pos', 'mean'),
        sent_neg=('neg', 'mean'),
        news_count=('compound', 'count'),
    ).reindex(business_days).fillna({'news_count': 0,
                                     'sent_compound': 0,
                                     'sent_pos': 0,
                                     'sent_neg': 0})
    agg['sector'] = sector
    return agg.reset_index()


def fetch_news_sentiment(df, start=START_DATE, end=END_DATE, max_threads=50):
    """Full GDELT news sentiment pipeline (disabled by default)."""
    if not USE_NEWS_SENTIMENT:
        print('News sentiment disabled (USE_NEWS_SENTIMENT=False). Returning empty.')
        return pd.DataFrame()

    from concurrent.futures import ThreadPoolExecutor as TPE

    business_days = pd.bdate_range(start, end)
    chunks = []
    dt = pd.Timestamp(start)
    end_ts = pd.Timestamp(end)
    while dt < end_ts:
        chunks.append((dt, min(dt + pd.Timedelta(days=90), end_ts)))
        dt += pd.Timedelta(days=90)

    def fetch_sector(sector):
        keywords = SECTOR_KEYWORDS.get(sector, [])
        records = []
        with TPE(max_workers=min(max_threads, len(keywords) * len(chunks))) as ex:
            futs = [
                ex.submit(_gdelt_chunk, kw, cf, ct)
                for kw in keywords
                for cf, ct in chunks
            ]
            for f in futs:
                records.extend(f.result())
        return _score_and_aggregate(records, sector, business_days)

    all_sent = []
    for sector in TICKERS_BY_SECTOR.keys():
        s = fetch_sector(sector)
        if not s.empty:
            all_sent.append(s)

    if not all_sent:
        return pd.DataFrame()
    return pd.concat(all_sent, ignore_index=True)

In [ ]:
# Cell 8 — 5: Assemble feature matrix

def assemble_features(df, sentiment_df=None):
    """
    Assemble the full feature matrix:
      1. Add sector_id + month
      2. Cross-asset: btc_momentum
      3. Macro: market_breadth_{sw} + sector_avg_rsi_{w}
      4. Optional: merge news sentiment
      5. Drop NaN rows (based on largest indicator windows)
    """
    df = df.copy()

    # Sector ID + month
    df['sector']    = df['ticker'].map(TICKER_TO_SECTOR)
    df['sector_id'] = df['sector'].map(SECTOR_LABELS).astype(float)
    df['month']     = df['Date'].dt.month.astype(float)

    # BTC momentum as cross-asset feature
    btc = df[df['ticker'] == 'BTC-USD'][['Date', 'momentum_20d']].rename(
        columns={'momentum_20d': 'btc_momentum'}
    )
    df = df.merge(btc, on='Date', how='left')
    df['btc_momentum'] = df['btc_momentum'].fillna(0.0)

    # Market breadth: fraction of tickers above their SMA (per Date)
    for sw in SMA_WINDOWS:
        breadth = (
            df.groupby('Date')[f'sma_ratio_{sw}']
            .apply(lambda x: (x > 1.0).mean())
            .rename(f'market_breadth_{sw}')
        )
        df = df.merge(breadth, on='Date', how='left')
        df[f'market_breadth_{sw}'] = df[f'market_breadth_{sw}'].fillna(0.5)

    # Sector average RSI
    for rw in RSI_WINDOWS:
        df[f'sector_avg_rsi_{rw}'] = df.groupby(['Date', 'sector'])[f'rsi_{rw}'].transform('mean')

    # Optional news sentiment merge
    if USE_NEWS_SENTIMENT and sentiment_df is not None and not sentiment_df.empty:
        df = df.merge(sentiment_df, on=['Date', 'sector'], how='left')
        for col in NEWS_FEATURE_COLS:
            df[col] = df[col].fillna(0.0)

    # Remove rows with NaN in worst-case window features (RSI=21, BB=25, SMA=70)
    worst_case_cols = build_feature_cols(21, 25, 70)
    # Also include all precomputed variants
    all_indicator_cols = [c for c in worst_case_cols if c in df.columns]
    df = df.dropna(subset=all_indicator_cols)

    print(f'Feature matrix assembled. Shape: {df.shape}, '
          f'positive rate: {df["target"].mean():.1%}')
    return df.reset_index(drop=True)

In [ ]:
# Cell 9 — 6: Optuna optimisation (base XGBoost)

def optimize_xgb(df_train, n_trials=N_OPTUNA_TRIALS, seed_params=SEED_PARAMS):
    """
    Hyperparameter optimisation via Optuna with Walk-Forward temporal CV.
    Returns best_params dict.
    """
    # Sort dates and prepare walk-forward split indices
    all_dates = np.sort(df_train['Date'].unique())
    n_dates   = len(all_dates)
    min_train = int(n_dates * 0.40)
    fold_size = (n_dates - min_train) // N_WF_SPLITS

    date_to_idx = {d: i for i, d in enumerate(all_dates)}
    df_train = df_train.copy()
    df_train['_date_idx'] = df_train['Date'].map(date_to_idx)

    def objective(trial):
        rsi_w = trial.suggest_categorical('rsi_window', RSI_WINDOWS)
        bb_w  = trial.suggest_categorical('bb_window',  BB_WINDOWS)
        sma_w = trial.suggest_categorical('sma_window', SMA_WINDOWS)
        feat_cols = build_feature_cols(rsi_w, bb_w, sma_w)

        grow_policy = trial.suggest_categorical('grow_policy', ['depthwise', 'lossguide'])
        max_leaves  = trial.suggest_int('max_leaves', 31, 1024)
        if grow_policy == 'depthwise':
            max_leaves = 0

        params = dict(
            grow_policy      = grow_policy,
            max_leaves       = max_leaves,
            max_bin          = trial.suggest_categorical('max_bin', [64, 128, 256]),
            max_depth        = trial.suggest_int('max_depth', 3, 15),
            min_child_weight = trial.suggest_int('min_child_weight', 5, 100),
            gamma            = trial.suggest_float('gamma', 0.0, 10.0),
            reg_alpha        = trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
            reg_lambda       = trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
            learning_rate    = trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            n_estimators     = trial.suggest_int('n_estimators', 100, 1500),
            subsample        = trial.suggest_float('subsample', 0.5, 0.9),
            colsample_bytree = trial.suggest_float('colsample_bytree', 0.2, 0.7),
        )
        focal_gamma = trial.suggest_float('focal_gamma', 0.5, 5.0)
        focal_alpha = trial.suggest_float('focal_alpha', 0.05, 0.5)

        focal_obj = make_focal_objective(focal_gamma, focal_alpha)

        fold_aps = []
        for fold_i in range(N_WF_SPLITS):
            train_end   = min_train + fold_i * fold_size
            val_end     = min_train + (fold_i + 1) * fold_size
            if val_end > n_dates:
                break

            train_mask = df_train['_date_idx'] < train_end
            val_mask   = (df_train['_date_idx'] >= train_end) & (df_train['_date_idx'] < val_end)

            X_tr = df_train.loc[train_mask, feat_cols].values.astype(np.float32)
            y_tr = df_train.loc[train_mask, 'target'].values.astype(np.int8)
            X_val = df_train.loc[val_mask, feat_cols].values.astype(np.float32)
            y_val = df_train.loc[val_mask, 'target'].values.astype(np.int8)

            if X_tr.shape[0] < 50 or X_val.shape[0] < 10:
                continue
            if y_val.sum() < 2:
                continue

            # Inner 90/10 split for early stopping
            rng_es = np.random.RandomState(RANDOM_STATE + fold_i)
            perm   = rng_es.permutation(len(X_tr))
            n_fit  = int(len(perm) * 0.9)
            fit_idx, es_idx = perm[:n_fit], perm[n_fit:]

            dtrain = xgb.DMatrix(X_tr[fit_idx], label=y_tr[fit_idx])
            des    = xgb.DMatrix(X_tr[es_idx],  label=y_tr[es_idx])
            dval   = xgb.DMatrix(X_val, label=y_val)

            bst = xgb.train(
                {**params, 'tree_method': 'hist', 'seed': RANDOM_STATE, 'disable_default_eval_metric': 1},
                dtrain,
                num_boost_round=params['n_estimators'],
                obj=focal_obj,
                evals=[(des, 'es')],
                custom_metric=lambda p, d: ('logloss',
                    float(np.mean(-d.get_label() * np.log(np.clip(1/(1+np.exp(-p)), 1e-7, 1-1e-7))
                              - (1-d.get_label()) * np.log(np.clip(1/(1+np.exp(p)), 1e-7, 1-1e-7))))),
                early_stopping_rounds=EARLY_STOPPING_ROUNDS,
                verbose_eval=False,
            )

            raw_preds = bst.predict(dval)
            probs = 1.0 / (1.0 + np.exp(-raw_preds))
            if y_val.sum() >= 2:
                fold_aps.append(average_precision_score(y_val, probs))

            # Optuna pruning
            trial.report(np.mean(fold_aps) if fold_aps else 0.0, fold_i)
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()

        return np.mean(fold_aps) if fold_aps else 0.0

    sampler = optuna.samplers.TPESampler(multivariate=True, constant_liar=True, seed=42)
    pruner  = optuna.pruners.MedianPruner(n_startup_trials=10, n_warmup_steps=1)
    study   = optuna.create_study(direction='maximize', sampler=sampler, pruner=pruner)

    # Seed trial with V8 best params
    study.enqueue_trial(seed_params)

    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

    best = study.best_params
    print(f'\nBest trial: AP={study.best_value:.4f}')
    print(f'Best params: {best}')
    return best

In [ ]:
# Cell 10 — 7: Holdout plot function

def plot_holdout_results(df_final, final_tickers, filtered_signals, title='FINAL Holdout'):
    """
    For each ticker in final_tickers: plot close price, rally periods,
    target days, and model buy signals.
    """
    n = len(final_tickers)
    fig, axes = plt.subplots(n, 1, figsize=(18, 5 * n))
    if n == 1:
        axes = [axes]

    for ax, ticker in zip(axes, final_tickers):
        sub = df_final[df_final['ticker'] == ticker].sort_values('Date')
        if sub.empty:
            ax.set_title(f'{ticker} — no data')
            continue

        dates = sub['Date'].values
        close = sub['close'].values

        # Close price
        ax.plot(dates, close, color='black', linewidth=0.8, label='Close')

        # Rally shading
        rally_mask = sub['rally'].values == 1
        in_rally = False
        rally_start = None
        for i, (d, r) in enumerate(zip(dates, rally_mask)):
            if r and not in_rally:
                rally_start = d
                in_rally = True
            elif not r and in_rally:
                ax.axvspan(rally_start, dates[i - 1], alpha=0.15, color='green')
                in_rally = False
        if in_rally:
            ax.axvspan(rally_start, dates[-1], alpha=0.15, color='green')

        # Target days (orange dots)
        target_mask = sub['target'].values == 1
        ax.scatter(dates[target_mask], close[target_mask],
                   color='orange', s=20, zorder=3, label='Target day')

        # Model signals (red triangles)
        sig_dates = filtered_signals.get(ticker, np.array([], dtype='datetime64[ns]'))
        if len(sig_dates) > 0:
            sig_sub = sub[sub['Date'].isin(sig_dates)]
            ax.scatter(sig_sub['Date'].values, sig_sub['close'].values,
                       marker='^', color='red', s=60, zorder=4, label='Buy signal')

        ax.set_title(f"{ticker} — {COMPANY_NAMES.get(ticker, ticker)}", fontsize=10)
        ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
        ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
        ax.tick_params(axis='x', rotation=45, labelsize=7)
        ax.legend(loc='upper left', fontsize=7)
        ax.set_ylabel('Price', fontsize=8)

    fig.suptitle(title, fontsize=14, y=1.001)
    plt.tight_layout()
    plt.show()

In [ ]:
# Cell 11 — 8: Run pipeline
# Orchestrates: Download → Target → Indicators → Feature Assembly → 3-Way Split

# ── 1. Load data ────────────────────────────────────────────────────────────
df_raw = load_stock_data()

# ── 2. Target vector ────────────────────────────────────────────────────────
df_with_target = create_target(df_raw)

# ── 3. Technical indicators ─────────────────────────────────────────────────
df_with_indicators = add_technical_indicators(df_with_target)

# ── 4. News sentiment (disabled) ────────────────────────────────────────────
sentiment_df = fetch_news_sentiment(df_with_indicators)

# ── 5. Assemble feature matrix ──────────────────────────────────────────────
df_features = assemble_features(df_with_indicators, sentiment_df)

# ── 6. 3-Way stratified ticker split ────────────────────────────────────────
rng_split = np.random.RandomState(42)

test_tickers  = []
final_tickers = []
train_tickers = []

for sector, tickers in TICKERS_BY_SECTOR.items():
    # Keep only tickers that actually exist in the data
    available = [t for t in tickers if t in df_features['ticker'].unique()]
    rng_split.shuffle(available)
    n = len(available)
    if n >= 3:
        test_tickers.append(available[0])
        final_tickers.append(available[1])
        train_tickers.extend(available[2:])
    elif n == 2:
        test_tickers.append(available[0])
        train_tickers.append(available[1])
    elif n == 1:
        train_tickers.extend(available)

# Fill TEST and FINAL to ~10 each from remaining TRAIN pool
remaining_train = [t for t in train_tickers]
rng_split.shuffle(remaining_train)
ptr = 0
while len(test_tickers) < 10 and ptr < len(remaining_train):
    t = remaining_train[ptr]
    if t not in test_tickers and t not in final_tickers:
        test_tickers.append(t)
        train_tickers.remove(t)
    ptr += 1

rng_split.shuffle(remaining_train)
ptr = 0
while len(final_tickers) < 10 and ptr < len(remaining_train):
    t = remaining_train[ptr]
    if t not in test_tickers and t not in final_tickers and t in train_tickers:
        final_tickers.append(t)
        train_tickers.remove(t)
    ptr += 1

# Ensure no overlap
test_set  = set(test_tickers)
final_set = set(final_tickers)
train_tickers = [t for t in train_tickers if t not in test_set and t not in final_set]

print(f'TRAIN: {len(train_tickers)} tickers — {train_tickers}')
print(f'TEST:  {len(test_tickers)} tickers — {test_tickers}')
print(f'FINAL: {len(final_tickers)} tickers — {final_tickers}')

df_train = df_features[df_features['ticker'].isin(train_tickers)].copy()
df_test  = df_features[df_features['ticker'].isin(test_tickers)].copy()
df_final = df_features[df_features['ticker'].isin(final_tickers)].copy()

print(f'\nSplit sizes — TRAIN: {len(df_train):,}  TEST: {len(df_test):,}  FINAL: {len(df_final):,}')

In [ ]:
# Cell 12 — 8b: Optuna + final model (Phases 1–5)
# Phase 1: Optuna base, Phase 2-3: Base models + SHAP + Meta features
# Phase 4: Meta-Learner Optuna, Phase 5: Calibration + Threshold

import mlflow
try:
    mlflow.autolog(disable=True)
except Exception:
    pass

# ── Phase 1: Optuna HP optimisation on TRAIN ────────────────────────────────
print('=' * 60)
print('Phase 1: Optuna base-model HP optimisation')
print('=' * 60)
best_params = optimize_xgb(df_train)

rsi_w = best_params['rsi_window']
bb_w  = best_params['bb_window']
sma_w = best_params['sma_window']
FEAT_COLS = build_feature_cols(rsi_w, bb_w, sma_w)
print(f'\nUsing features: RSI={rsi_w}, BB={bb_w}, SMA={sma_w}  ({len(FEAT_COLS)} features)')

focal_gamma = best_params['focal_gamma']
focal_alpha = best_params['focal_alpha']
focal_obj   = make_focal_objective(focal_gamma, focal_alpha)
focal_obj_lgb = make_focal_objective_lgb(focal_gamma, focal_alpha)

xgb_base_params = {
    k: v for k, v in best_params.items()
    if k not in ('rsi_window', 'bb_window', 'sma_window', 'focal_gamma', 'focal_alpha')
}
xgb_base_params['tree_method'] = 'hist'

lgb_params = dict(
    max_depth        = best_params['max_depth'],
    num_leaves       = min(best_params.get('max_leaves', 127), 255),
    learning_rate    = best_params['learning_rate'],
    subsample        = best_params['subsample'],
    colsample_bytree = best_params['colsample_bytree'],
    min_child_weight = best_params['min_child_weight'],
    reg_alpha        = best_params['reg_alpha'],
    reg_lambda       = best_params['reg_lambda'],
    n_estimators     = best_params['n_estimators'],
    verbose          = -1,
)

X_train_all = df_train[FEAT_COLS].values.astype(np.float32)
y_train_all = df_train['target'].values.astype(np.int8)

# ── Phase 2: Train 6 base models ────────────────────────────────────────────
print('\n' + '=' * 60)
print('Phase 2: Training 6 base models')
print('=' * 60)
base_models = []  # list of (name, model_or_booster, model_type)

def _bootstrap_split(seed_i, X, y):
    """Bootstrap + ES split. Returns (X_fit, y_fit, X_es, y_es)."""
    rng_boot = np.random.RandomState(seed_i)
    rng_es   = np.random.RandomState(seed_i + 100)
    boot_idx = rng_boot.choice(len(X), size=len(X), replace=True)
    perm     = rng_es.permutation(len(boot_idx))
    n_fit    = int(len(perm) * 0.9)
    fit_idx  = boot_idx[perm[:n_fit]]
    es_idx   = boot_idx[perm[n_fit:]]
    return X[fit_idx], y[fit_idx], X[es_idx], y[es_idx]

# Model 1 & 2: XGBoost
for m_idx, seed_i in enumerate([RANDOM_STATE, RANDOM_STATE + 1]):
    name = f'XGB-{m_idx+1}'
    print(f'  Training {name}...')
    X_fit, y_fit, X_es, y_es = _bootstrap_split(seed_i, X_train_all, y_train_all)
    dtrain_m = xgb.DMatrix(X_fit, label=y_fit)
    des_m    = xgb.DMatrix(X_es,  label=y_es)
    p = {**xgb_base_params, 'seed': seed_i, 'disable_default_eval_metric': 1}
    bst = xgb.train(
        p, dtrain_m,
        num_boost_round=xgb_base_params['n_estimators'],
        obj=focal_obj,
        evals=[(des_m, 'es')],
        custom_metric=lambda pr, d: ('logloss',
            float(np.mean(-d.get_label() * np.log(np.clip(1/(1+np.exp(-pr)), 1e-7, 1-1e-7))
                      - (1-d.get_label()) * np.log(np.clip(1/(1+np.exp(pr)), 1e-7, 1-1e-7))))),
        early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        verbose_eval=False,
    )
    base_models.append((name, bst, 'xgb'))
    print(f'  {name} done — best iteration: {bst.best_iteration}')

# Model 3 & 4: LightGBM
for m_idx, seed_i in enumerate([RANDOM_STATE + 10, RANDOM_STATE + 11]):
    name = f'LGB-{m_idx+1}'
    print(f'  Training {name}...')
    X_fit, y_fit, X_es, y_es = _bootstrap_split(seed_i, X_train_all, y_train_all)
    dtrain_lgb = lgb.Dataset(X_fit, label=y_fit)
    des_lgb    = lgb.Dataset(X_es,  label=y_es, reference=dtrain_lgb)
    callbacks  = [lgb.early_stopping(EARLY_STOPPING_ROUNDS, verbose=False), lgb.log_evaluation(-1)]
    p_lgb = {**lgb_params, 'seed': seed_i, 'verbosity': -1}
    n_est = p_lgb.pop('n_estimators', 300)
    bst_lgb = lgb.train(
        p_lgb, dtrain_lgb,
        num_boost_round=n_est,
        fobj=focal_obj_lgb,
        valid_sets=[des_lgb],
        callbacks=callbacks,
    )
    base_models.append((name, bst_lgb, 'lgb'))
    print(f'  {name} done — best iteration: {bst_lgb.best_iteration}')

# Model 5: Random Forest
print('  Training RF...')
rf = RandomForestClassifier(
    n_estimators=500,
    max_depth=best_params['max_depth'],
    min_samples_leaf=20,
    max_features='sqrt',
    class_weight='balanced_subsample',
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
rf.fit(X_train_all, y_train_all)
base_models.append(('RF', rf, 'rf'))
print('  RF done.')

# Model 6: TabPFN
if TABPFN_AVAILABLE:
    print('  Training TabPFN...')
    try:
        rng_tab = np.random.RandomState(RANDOM_STATE)
        n_total = len(X_train_all)
        n_pos   = int(y_train_all.sum())
        n_neg   = n_total - n_pos
        n_sub   = 10000
        n_pos_s = int(n_sub * n_pos / n_total)
        n_neg_s = n_sub - n_pos_s
        pos_idx = rng_tab.choice(np.where(y_train_all == 1)[0], size=min(n_pos_s, n_pos), replace=False)
        neg_idx = rng_tab.choice(np.where(y_train_all == 0)[0], size=min(n_neg_s, n_neg), replace=False)
        sub_idx = np.concatenate([pos_idx, neg_idx])
        tabpfn  = TabPFNClassifier(N_ensemble_configurations=8, device='cpu')
        tabpfn.fit(X_train_all[sub_idx], y_train_all[sub_idx])
        base_models.append(('TabPFN', tabpfn, 'tabpfn'))
        print('  TabPFN done.')
    except Exception as e:
        print(f'  TabPFN failed ({e}), continuing without it.')
else:
    print('  TabPFN not available, skipping.')

print(f'\nBase models ready: {[m[0] for m in base_models]}')

# ── Predict helper ───────────────────────────────────────────────────────────
def _predict_base(model_tuple, X):
    name, model, mtype = model_tuple
    if mtype == 'xgb':
        raw = model.predict(xgb.DMatrix(X))
        return 1.0 / (1.0 + np.exp(-raw))
    elif mtype == 'lgb':
        raw = model.predict(X)
        return 1.0 / (1.0 + np.exp(-raw))
    elif mtype in ('rf', 'tabpfn'):
        return model.predict_proba(X)[:, 1]

# ── Phase 3: SHAP feature selection + meta feature matrix ────────────────────
print('\n' + '=' * 60)
print('Phase 3: SHAP feature selection')
print('=' * 60)

X_test_feat  = df_test[FEAT_COLS].values.astype(np.float32)
y_test       = df_test['target'].values.astype(np.int8)
X_final_feat = df_final[FEAT_COLS].values.astype(np.float32)
y_final      = df_final['target'].values.astype(np.int8)

# SHAP on XGB-1, 2000 samples from TEST
rng_shap   = np.random.RandomState(RANDOM_STATE)
shap_idx   = rng_shap.choice(len(X_test_feat), size=min(2000, len(X_test_feat)), replace=False)
xgb_model_1 = base_models[0][1]  # first XGBoost booster
explainer   = shap.TreeExplainer(xgb_model_1)
shap_vals   = explainer.shap_values(xgb.DMatrix(X_test_feat[shap_idx]))
mean_shap   = np.abs(shap_vals).mean(axis=0)
top5_idx    = np.argsort(mean_shap)[::-1][:5]
top5_names  = [FEAT_COLS[i] for i in top5_idx]
rename_map  = build_rename_map(rsi_w, bb_w, sma_w)
print(f'Top 5 SHAP features: {[rename_map.get(n, n) for n in top5_names]}')

# Build meta-feature matrix: 6 base predictions + top-5 features
def build_meta_features(X_feat):
    base_preds = np.column_stack([
        _predict_base(m, X_feat) for m in base_models
    ])
    top5_feats = X_feat[:, top5_idx]
    return np.hstack([base_preds, top5_feats]).astype(np.float32)

X_meta_test  = build_meta_features(X_test_feat)
X_meta_final = build_meta_features(X_final_feat)
print(f'Meta-feature matrix shape: TEST={X_meta_test.shape}, FINAL={X_meta_final.shape}')

# ── Phase 4: Meta-Learner Optuna ─────────────────────────────────────────────
print('\n' + '=' * 60)
print('Phase 4: Meta-Learner Optuna')
print('=' * 60)

N_META_FOLDS = 3
all_dates_test = np.sort(df_test['Date'].unique())
n_meta_dates   = len(all_dates_test)
meta_min_train = int(n_meta_dates * 0.40)
meta_fold_size = (n_meta_dates - meta_min_train) // N_META_FOLDS
date_to_idx_test = {d: i for i, d in enumerate(all_dates_test)}
df_test_idx  = df_test['Date'].map(date_to_idx_test).values

def meta_objective(trial):
    params = dict(
        max_depth        = trial.suggest_int('max_depth', 2, 6),
        min_child_weight = trial.suggest_int('min_child_weight', 10, 200),
        gamma            = trial.suggest_float('gamma', 0.0, 5.0),
        reg_alpha        = trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        reg_lambda       = trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        learning_rate    = trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
        n_estimators     = trial.suggest_int('n_estimators', 50, 500),
        subsample        = trial.suggest_float('subsample', 0.5, 0.9),
        colsample_bytree = trial.suggest_float('colsample_bytree', 0.3, 1.0),
        scale_pos_weight = trial.suggest_float('scale_pos_weight', 1.0, 15.0),
        tree_method      = 'hist',
        use_label_encoder= False,
        eval_metric      = 'aucpr',
        seed             = RANDOM_STATE,
    )

    fold_aps = []
    for fold_i in range(N_META_FOLDS):
        train_end = meta_min_train + fold_i * meta_fold_size
        val_end   = meta_min_train + (fold_i + 1) * meta_fold_size
        if val_end > n_meta_dates:
            break

        tr_mask  = df_test_idx < train_end
        val_mask = (df_test_idx >= train_end) & (df_test_idx < val_end)

        X_tr  = X_meta_test[tr_mask]
        y_tr  = y_test[tr_mask]
        X_val = X_meta_test[val_mask]
        y_val = y_test[val_mask]

        if X_tr.shape[0] < 20 or y_val.sum() < 2:
            continue

        rng_m = np.random.RandomState(RANDOM_STATE)
        perm  = rng_m.permutation(len(X_tr))
        n_fit = int(len(perm) * 0.85)

        clf = xgb.XGBClassifier(**params)
        clf.fit(
            X_tr[perm[:n_fit]], y_tr[perm[:n_fit]],
            eval_set=[(X_tr[perm[n_fit:]], y_tr[perm[n_fit:]])],
            early_stopping_rounds=20,
            verbose=False,
        )
        probs = clf.predict_proba(X_val)[:, 1]
        if y_val.sum() >= 2:
            fold_aps.append(average_precision_score(y_val, probs))

    return np.mean(fold_aps) if fold_aps else 0.0

meta_sampler = optuna.samplers.TPESampler(multivariate=True, constant_liar=True, seed=42)
meta_study   = optuna.create_study(direction='maximize', sampler=meta_sampler)
meta_study.optimize(meta_objective, n_trials=N_META_TRIALS, show_progress_bar=True)

meta_best = {**meta_study.best_params,
             'tree_method': 'hist',
             'use_label_encoder': False,
             'eval_metric': 'aucpr',
             'seed': RANDOM_STATE}
print(f'\nMeta-Learner best AP={meta_study.best_value:.4f}')

# Final meta-model trained on all TEST data
rng_final_meta = np.random.RandomState(RANDOM_STATE)
perm_final     = rng_final_meta.permutation(len(X_meta_test))
n_fit_final    = int(len(perm_final) * 0.9)
meta_clf = xgb.XGBClassifier(**meta_best)
meta_clf.fit(
    X_meta_test[perm_final[:n_fit_final]],  y_test[perm_final[:n_fit_final]],
    eval_set=[(X_meta_test[perm_final[n_fit_final:]], y_test[perm_final[n_fit_final:]])],
    early_stopping_rounds=20,
    verbose=False,
)
print('Final meta-model trained.')

# ── Phase 5a: OOF predictions on TEST ──────────────────────────────────────
print('\n' + '=' * 60)
print('Phase 5: Calibration + Threshold')
print('=' * 60)

y_prob_test_raw = np.zeros(len(y_test), dtype=np.float64)

# CV splits (reuse meta_min_train / meta_fold_size)
meta_cv_splits = []
for fold_i in range(N_META_FOLDS):
    train_end = meta_min_train + fold_i * meta_fold_size
    val_end   = meta_min_train + (fold_i + 1) * meta_fold_size
    if val_end > n_meta_dates:
        break
    tr_mask  = df_test_idx < train_end
    val_mask = (df_test_idx >= train_end) & (df_test_idx < val_end)
    meta_cv_splits.append((np.where(tr_mask)[0], np.where(val_mask)[0]))

for trn_idx, val_idx in meta_cv_splits:
    if len(trn_idx) < 20 or len(val_idx) < 5:
        continue
    rng_cv = np.random.RandomState(RANDOM_STATE)
    perm_cv = rng_cv.permutation(len(trn_idx))
    n_fit_cv = int(len(perm_cv) * 0.85)
    clf_cv = xgb.XGBClassifier(**meta_best)
    clf_cv.fit(
        X_meta_test[trn_idx[perm_cv[:n_fit_cv]]],
        y_test[trn_idx[perm_cv[:n_fit_cv]]],
        eval_set=[(X_meta_test[trn_idx[perm_cv[n_fit_cv:]]],
                   y_test[trn_idx[perm_cv[n_fit_cv:]]])],
        early_stopping_rounds=20,
        verbose=False,
    )
    y_prob_test_raw[val_idx] = clf_cv.predict_proba(X_meta_test[val_idx])[:, 1]

# For indices not covered by CV, use final meta-model
covered = np.concatenate([val for _, val in meta_cv_splits]) if meta_cv_splits else np.array([], dtype=int)
uncovered = np.setdiff1d(np.arange(len(y_test)), covered)
if len(uncovered) > 0:
    y_prob_test_raw[uncovered] = meta_clf.predict_proba(X_meta_test[uncovered])[:, 1]

# FINAL predictions from final meta-model
y_prob_final_raw = meta_clf.predict_proba(X_meta_final)[:, 1]

# ── Phase 5b: Isotonic calibration ─────────────────────────────────────────
iso_cal = IsotonicRegression(out_of_bounds='clip')
iso_cal.fit(y_prob_test_raw, y_test)
y_prob_test  = iso_cal.transform(y_prob_test_raw)
y_prob_final = iso_cal.transform(y_prob_final_raw)
print('Isotonic calibration fitted.')

# ── Phase 5c: Threshold optimisation ───────────────────────────────────────
def find_precision_threshold(y_true, y_prob, target_prec=0.60, min_signals=5):
    thresholds = np.arange(0.10, 0.96, 0.01)
    best_thresh = None
    for thr in thresholds:
        preds = (y_prob >= thr).astype(int)
        n_sig = preds.sum()
        if n_sig < min_signals:
            continue
        prec = precision_score(y_true, preds, zero_division=0)
        if prec >= target_prec:
            best_thresh = thr
            break
    return best_thresh

prec_arr, rec_arr, thr_arr = precision_recall_curve(y_test, y_prob_test)
f1_arr = 2 * prec_arr * rec_arr / (prec_arr + rec_arr + 1e-10)
f1_thresh = thr_arr[np.argmax(f1_arr[:-1])]
prec_thresh = find_precision_threshold(y_test, y_prob_test)

if prec_thresh is None:
    print('Warning: could not reach 60% precision on TEST. Falling back to F1-optimal threshold.')
    best_threshold = float(f1_thresh)
else:
    best_threshold = float(prec_thresh)

print(f'F1-optimal threshold:        {f1_thresh:.3f}')
print(f'Precision-targeted threshold: {best_threshold:.3f}')

# Store predictions in DataFrames for downstream analysis
df_test['prob']  = y_prob_test
df_final['prob'] = y_prob_final
print('\nPhase 5 complete.')

In [ ]:
# Cell 13 — 8d: Threshold analysis

# ── PR curves ────────────────────────────────────────────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

prec_test, rec_test, _ = precision_recall_curve(y_test, y_prob_test)
ap_test = average_precision_score(y_test, y_prob_test)
ax1.plot(rec_test, prec_test, color='steelblue', label=f'TEST (AP={ap_test:.3f})')

prec_fin, rec_fin, _ = precision_recall_curve(y_final, y_prob_final)
ap_fin = average_precision_score(y_final, y_prob_final)
ax1.plot(rec_fin, prec_fin, color='tomato', linestyle='--', label=f'FINAL (AP={ap_fin:.3f})')
ax1.axhline(0.60, color='gray', linestyle=':', linewidth=0.8, label='60% precision')
ax1.set_xlabel('Recall'); ax1.set_ylabel('Precision')
ax1.set_title('Precision-Recall Curve'); ax1.legend(); ax1.set_xlim([0, 1]); ax1.set_ylim([0, 1])

# Threshold vs Precision / Recall / F1 on TEST
thr_sweep = np.arange(0.01, 1.0, 0.01)
prec_sw = []; rec_sw = []; f1_sw = []
for t in thr_sweep:
    p_ = (y_prob_test >= t).astype(int)
    prec_sw.append(precision_score(y_test, p_, zero_division=0))
    rec_sw.append(recall_score(y_test, p_, zero_division=0))
    f1_sw.append(f1_score(y_test, p_, zero_division=0))

ax2.plot(thr_sweep, prec_sw, color='tomato', label='Precision')
ax2.plot(thr_sweep, rec_sw,  color='steelblue', label='Recall')
ax2.plot(thr_sweep, f1_sw,   color='green', linestyle='--', label='F1')
ax2.axvline(best_threshold, color='black', linewidth=1.2,
            label=f'Best thr={best_threshold:.2f}')
ax2.axvline(f1_thresh, color='purple', linestyle=':', linewidth=1.0,
            label=f'F1-opt={f1_thresh:.2f}')
ax2.set_xlabel('Threshold'); ax2.set_ylabel('Score')
ax2.set_title('TEST — Threshold vs Metrics'); ax2.legend()
ax2.set_xlim([0, 1]); ax2.set_ylim([0, 1])
plt.tight_layout()
plt.show()

# ── TEST raw threshold sweep table ──────────────────────────────────────────
print('\n── TEST (raw) Threshold Sweep ──')
rows_test = []
for t in np.arange(0.05, 0.96, 0.05):
    preds = (y_prob_test >= t).astype(int)
    n_sig = preds.sum()
    tp    = int((preds * y_test).sum())
    fp    = int(preds.sum() - tp)
    prec  = precision_score(y_test, preds, zero_division=0)
    rec   = recall_score(y_test, preds, zero_division=0)
    f1    = f1_score(y_test, preds, zero_division=0)
    rows_test.append({'Thr': f'{t:.2f}', 'Prec': f'{prec:.2%}', 'Rec': f'{rec:.2%}',
                      'F1': f'{f1:.3f}', 'Signals': n_sig, 'TP': tp, 'FP': fp})
print(pd.DataFrame(rows_test).to_string(index=False))

# ── Signal filters ───────────────────────────────────────────────────────────
def apply_signal_filters(df_ticker_prob, threshold):
    """
    Apply consecutive filter (CONSECUTIVE_DAYS=1) then cooldown filter.
    Returns array of signal indices.
    """
    df_s = df_ticker_prob.sort_values('Date').reset_index(drop=True)
    raw  = (df_s['prob'].values >= threshold).astype(np.int8)
    n    = len(raw)

    # Consecutive filter
    consec = np.zeros(n, dtype=np.int8)
    for i in range(2, n):
        if raw[i-2] + raw[i-1] + raw[i] >= CONSECUTIVE_DAYS:
            consec[i] = 1

    # Cooldown filter
    final  = np.zeros(n, dtype=np.int8)
    last_signal = -999
    for i in range(n):
        if consec[i] == 1 and (i - last_signal) >= SIGNAL_COOLDOWN_DAYS:
            final[i] = 1
            last_signal = i

    return df_s.loc[final == 1, 'Date'].values


# ── FINAL filtered threshold sweep table ─────────────────────────────────────
print('\n── FINAL (filtered) Threshold Sweep ──')
rows_final = []
for t in np.arange(0.05, 0.96, 0.05):
    sig_dates_by_ticker = {}
    for ticker in final_tickers:
        sub = df_final[df_final['ticker'] == ticker]
        sig_dates_by_ticker[ticker] = apply_signal_filters(sub, t)

    # Compute precision over all FINAL tickers
    n_sig = 0; tp = 0; fp = 0
    for ticker, sig_dates in sig_dates_by_ticker.items():
        sub = df_final[df_final['ticker'] == ticker]
        for d in sig_dates:
            row = sub[sub['Date'] == d]
            if row.empty:
                continue
            n_sig += 1
            if row['target'].values[0] == 1:
                tp += 1
            else:
                fp += 1
    if n_sig == 0:
        continue
    prec = tp / n_sig
    check = '✓' if prec >= 0.60 else ''
    rows_final.append({'Thr': f'{t:.2f}', 'Prec': f'{prec:.2%}',
                       'Signals': n_sig, 'TP': tp, 'FP': fp, '≥60%': check})
print(pd.DataFrame(rows_final).to_string(index=False))

# ── Summary ──────────────────────────────────────────────────────────────────
print('\n── Result Summary ──')
for label, y_true, y_prob_arr, tickers_list, df_part in [
        ('TEST (raw)',      y_test,  y_prob_test,  test_tickers,  df_test),
        ('FINAL (filtered)',y_final, y_prob_final, final_tickers, df_final),
]:
    if label.startswith('TEST'):
        preds = (y_prob_arr >= best_threshold).astype(int)
        n_sig = preds.sum(); tp = int((preds * y_true).sum())
        fp = n_sig - tp
        prec = precision_score(y_true, preds, zero_division=0)
    else:
        n_sig = tp = fp = 0
        for t in tickers_list:
            sub = df_part[df_part['ticker'] == t]
            for d in apply_signal_filters(sub, best_threshold):
                row = sub[sub['Date'] == d]
                if row.empty: continue
                n_sig += 1
                if row['target'].values[0] == 1: tp += 1
                else: fp += 1
        prec = tp / n_sig if n_sig > 0 else 0.0

    status = 'PASS ✓' if prec >= 0.60 else 'MISS ✗'
    print(f'  {label:20s} | Signals={n_sig:4d} | TP={tp:4d} | FP={fp:4d} | '
          f'Precision={prec:.1%} | {status}')

In [ ]:
# Cell 14 — 8c: Holdout plots (FINAL tickers)

# Build filtered signals dict for FINAL tickers
filtered_signals_final = {}
for ticker in final_tickers:
    sub = df_final[df_final['ticker'] == ticker]
    filtered_signals_final[ticker] = apply_signal_filters(sub, best_threshold)

# Visualise
plot_holdout_results(
    df_final,
    final_tickers,
    filtered_signals_final,
    title=f'FINAL Holdout — Threshold={best_threshold:.2f}'
)